# 2-motor: rest-resistance

Measure the motor's terminal resistance at rest over the rotor's angle, by the meter's route and by the board's own, and read the brush drop off the line through the stall laps, so the teardown and stage 3 have a graded number and its spread to compare against.

## Purpose

The terminal resistance of a brushed DC motor is the first number every later model leans on. The current a rail pushes into a locked rotor, the copper's temperature read as a change in resistance, the back-EMF taken out of a terminal reading under drive, and the share of a servo's heat that lands in the winding all start from it. The motor has no datasheet. Its registry entry in `oscnb/motors.py` carries only what can be counted, three slots, two brushes and a nine tooth pinion, and the resistance comes from here.

Two things make the number harder than one meter reading. The meter reads the winding through two brush contacts, and what the contacts add depends on where the rotor stopped, on which commutator segment sits under each brush and how it sits. So a reading belongs to the rotor's stop and not to the motor, and the first question is how wide the spread over the angle is and what shape it has. The second is that a brush contact does not behave like a resistor. A contact drop of some tenths of a volt that stays about the same as the current changes reads as a large resistance at the meter's milliamp and as a small one at an amp. A line of volts against amps over the locked rotor separates the two, the slope is the resistance and the intercept is the brush drop.

Two routes read the same motor at the same twelve angles. The ohms route is the meter on ohms across the tabs with the motor off the board. The stall route is the board's own current and terminal reading at 100 % duty into the locked rotor at the three 1S rail levels, with no meter in it. The stall route is the one `osc ident` can run on an assembled servo, so the ohms route grades it here, and the teardown at the end of the stage reads the copper with no brushes in the path, which is what splits the brushes' share from the winding's.

## Procedure

`capture.py` runs one route per argument, `ohms` or `stall`. Both step the rig's wrench bar through the twelve holes of its bolt circle, which hold the rotor by its pinion at angles 30 degrees apart, and prompt at each hole, so every reading belongs to a known rotor angle. The bar is clamped on the pinion and lifted off by its screw, and nothing pushes or pulls along the shaft, since a shaft pressed past its end play rides the brushes wrong and the motor is not the same motor after.

The ohms route puts the meter on ohms across the tabs with the leads off the board. The meter pushes about a milliamp through the winding and both contacts. The route starts with the lead zero, the meter's leads shorted together, a mark and a twenty second hold, and only then the prompt to put them on the motor's leads. At each hole the script writes a mark with the host time and lets the meter log for forty seconds, since the reading moves for a while after the rotor has been turned. The lead zero takes the probe leads out of the reading, and the two short leads soldered to the tabs stay in it.

The stall route puts the leads on the board's motor connector, with no meter on the motor. The board comes up on the DPS, and at each hole the script trims the rail to 3.3 V, 3.7 V and 4.2 V, the three 1S levels, and at each level runs one `osc sweep`, a half second torque-off baseline, 100 % duty for 300 ms in one direction, a second of rest and 300 ms in the other. At 100 % the bridge holds one leg high and one low for the whole lap, nothing switches, and the chip's settled means read DC quantities. The three levels keep the stall current under an amp. The meter stays out of this route, since its packets come slower than a cold lap lasts, so the volts are the board's own terminal taps. After each hole the script holds twenty seconds before it asks for the next.

Two bench rules from the stage README hold throughout. The bare motor runs for at most thirty seconds at a time with twenty seconds of rest after, so the copper stays within a degree or two of the can, and every stall lap is short enough to leave the winding cold, under a joule here, since a reading of the winding at an unknown temperature is not a reading. And the rotor is held by the pinion only, never pressed along its shaft.

1. The motor in the rig with two leads soldered to its tabs, the bar on the pinion at hole 0, clamp snug. The second motor of the batch is read with the meter at a few angles first, by hand, and those readings go into the Second motor section below.
2. The meter on ohms on its manual 220 ohm range, Bluetooth on, its leads shorted together, nothing else connected. `capture.py ohms`. After the lead zero's hold, at the prompt, the meter's leads onto the motor's two leads. At each prompt after that, clamp loose, bar to the next hole, clamp snug.
3. The leads onto the board's motor connector. The board on the DPS with the adapter on its bus at 3 Mbps, running the experiment image, so the laps are open loop with only the raw current trip and the bus dead-man as protections.
4. `capture.py stall`. The DPS comes on before the board's leads go on, because its output comes up at its previous setting for a moment. The bar steps as before.

## Data

Two folders under the capture, `ohms` and `stall`.

- `ohms/marks.csv`: `t_host`, `mark`. `leads shorted` where the lead zero's hold starts, then one row per hole, `hole 0` to `hole 11`, then `end`.
- `ohms/dmm.csv`: the meter's log through the route, the lead zero included. `t_host`, `function` (`ohm`), `value` in ohms, `raw` the packet.
- `stall/laps.csv`: one row per hole and level. `hole`, `volts_set` the rail asked for, `dps_read` the DPS output that reached it, `vsys_read` the rail the board read at rest before the lap, `t_start` and `t_end` the host time the sweep ran, `folder` the lap's folder.
- `stall/hole-K/<V>V/sweep.csv` and `meta.json`, with V one of 3.3, 3.7 and 4.2: the baseline and the two directions at that hole and level, one row per control tick at 20 kHz. Segment 0 is the baseline, 1 and 2 the two directions. `current_raw`, `vmotor_a`, `vmotor_b`, `vbus_raw` and `ntc_raw` are ADC counts, `duty_counts` the applied window in timer ticks. The lap rows do not carry `vref_raw`.

### Setup

The motor, the board, the dataset's two routes, and the stage 0 and stage 1 results every conversion below reads.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oscnb import b41t, boards, laps, motors, results, tel
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"   # the house colors
COLOR = {"ohms": BLUE, "stall": ORANGE}                                 # one color per route, fixed through the notebook
MOTOR = motors.MOTORS[motors.DEFAULT]                                   # the motor, with its counted parts
BOARD = boards.BOARDS[boards.DEFAULT]                                   # the board, with the parts' nominal values
CAPTURE = (Path("../../../telemetry") / f"{MOTOR.key}__{BOARD.key}" / "rest-resistance" / "capture-1").resolve()
OHMS, STALL = CAPTURE / "ohms", CAPTURE / "stall"
HOLES = 12                # the rig's bolt circle, 30 degrees per hole
TAIL_S = 20               # the last stretch of each hole's forty seconds, where the meter has settled
HEAD_S = 5                # the first stretch after the mark, where the reading is still moving
RAIL_GRADED_V = 3.7       # the stage 1 conversions hold from this rail up; the 3.3 V level is reported apart there
HOLD_S = 20               # the lead zero's hold after the `leads shorted` mark
LEAD_S = 10               # the hold's last stretch, where the meter has settled on the shorted leads
print(MOTOR.key, "on", BOARD.key, "from", results.rel(CAPTURE))
print(f"{MOTOR.label}: {MOTOR.slots} slots, {MOTOR.brushes} brushes, {MOTOR.ripple_per_rev} commutations per turn, a {MOTOR.pinion_teeth} tooth pinion")

DMM = results.read("0-instruments", "dmm")                            # the meter's classes and its packet interval
CS = results.read("1-board", "current-scale", pair=BOARD.key)          # counts per amp and the in-drive offset over the lap's own zero
TC = results.read("1-board", "terminal-chain", pair=BOARD.key)         # counts per volt of tap difference and its offset
G_CPA, B_OFF = CS["counts_per_a"]["value"], CS["drive_offset_counts"]["value"]
K_CPV, C_OFF = TC["terminal_counts_per_v"]["value"], TC["terminal_offset_counts"]["value"]
OHM_CLASS = DMM["accuracy"]["ohm_220"]
print(f"current: counts over the lap's zero = {B_OFF:+.1f} + {G_CPA:.1f} x amps ({CS['counts_per_a']['lo']:.1f} to {CS['counts_per_a']['hi']:.1f}), "
      f"graded over {CS['amps_range']['lo']:.2f} to {CS['amps_range']['hi']:.2f} A from {RAIL_GRADED_V} V up")
print(f"terminals: tap difference = {C_OFF:+.1f} + {K_CPV:.2f} x volts ({TC['terminal_counts_per_v']['lo']:.2f} to {TC['terminal_counts_per_v']['hi']:.2f})")
print(f"meter: ohms {OHM_CLASS['pct']['value']} % + {OHM_CLASS['fixed']['value']} ohm in {OHM_CLASS['step']['value']} ohm steps; "
      f"a packet every {DMM['packet_interval_s']['value']:.2f} s")

### ohms/marks.csv and ohms/dmm.csv

The meter's log through the ohms route. On the left the shorted leads through the lead zero's hold, the stretch the lead zero is taken from shaded. On the right every hole's mark drawn on the log and the tail window the reading is taken from shaded.

In [ ]:
def window(d, t_lo, t_hi):
    return d[(d["t_host"] >= t_lo) & (d["t_host"] < t_hi)]


marks = pd.read_csv(OHMS / "marks.csv")
dmm_ohms = b41t.load(OHMS / "dmm.csv")                                            # t_host, function, value in ohms, raw
t_short = float(marks.loc[marks["mark"] == "leads shorted", "t_host"].iloc[0])
shorted = window(dmm_ohms, t_short + HOLD_S - LEAD_S, t_short + HOLD_S)["value"].dropna()   # the hold's end, before the leads move to the motor
LEAD_ZERO, LEAD_LO, LEAD_HI = float(shorted.median()), float(shorted.quantile(0.25)), float(shorted.quantile(0.75))
holes = marks[marks["mark"].str.startswith("hole")].copy()
holes["hole"] = holes["mark"].str.split().str[1].astype(int)
t_end_ohms = float(marks.loc[marks["mark"] == "end", "t_host"].iloc[0])
holes["t_next"] = holes["t_host"].shift(-1).fillna(t_end_ohms)                    # each hole's window runs from its mark to the next
holes = holes.set_index("hole")
show(marks, n=None)
print(len(dmm_ohms), "packets,", dmm_ohms["function"].unique().tolist(), "range byte", dmm_ohms["raw"].astype(str).str[:4].unique().tolist(),
      f"over {dmm_ohms['t'].iloc[-1]:.0f} s; {len(holes)} holes, {holes['t_next'].sub(holes['t_host']).min():.0f} to {holes['t_next'].sub(holes['t_host']).max():.0f} s each")
print(f"lead zero: {LEAD_ZERO:.3f} ohm, quartiles {LEAD_LO:.3f} to {LEAD_HI:.3f}, from {len(shorted)} packets {HOLD_S - LEAD_S} to {HOLD_S} s after the mark; "
      f"the hole 0 mark came {holes['t_host'].iloc[0] - t_short:.0f} s after it")
t0 = dmm_ohms["t_host"].iloc[0]
inside = dmm_ohms[(dmm_ohms["t_host"] >= holes["t_host"].min()) & (dmm_ohms["t_host"] < t_end_ohms)]["value"].dropna()
fig, ax = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 3.5]})
w = window(dmm_ohms, t_short, holes["t_host"].iloc[0])
ax[0].plot(w["t_host"] - t_short, w["value"], ".", ms=3, color=COLOR["ohms"])
ax[0].axvspan(HOLD_S - LEAD_S, HOLD_S, color=MUTED, alpha=0.15)                     # the window the lead zero comes from
ax[0].axvline(HOLD_S, color=MUTED, lw=0.6)                                          # the hold ends, the prompt to move the leads
ax[0].axhline(LEAD_ZERO, color=INK, lw=0.8, label=f"lead zero {LEAD_ZERO:.3f} ohm")
ax[0].set_ylim(0, 2 * shorted.quantile(0.99) + 0.05)                                # the shorted leads' range; the motor after the move would swamp it
ax[0].set_xlabel("seconds after `leads shorted`"); ax[0].set_ylabel("meter (ohm)"); ax[0].set_title("the lead zero")
ax[0].legend(fontsize=8)
ax[1].plot(dmm_ohms["t"], dmm_ohms["value"], ".", ms=3, color=COLOR["ohms"])     # every packet
for k, h in holes.iterrows():
    ax[1].axvline(h["t_host"] - t0, color=MUTED, lw=0.6)                            # the mark
    ax[1].axvspan(h["t_next"] - TAIL_S - t0, h["t_next"] - t0, color=MUTED, alpha=0.15)  # the tail the reading comes from
    ax[1].annotate(str(k), (h["t_host"] - t0 + 1, 0.97), xycoords=("data", "axes fraction"), fontsize=8, color=MUTED)
ax[1].set_ylim(inside.quantile(0.01) * 0.97, inside.quantile(0.99) * 1.03)          # the holes' range; an open-lead packet at a move would swamp it
ax[1].set_xlabel("time from the first packet (s)"); ax[1].set_ylabel("meter (ohm)"); ax[1].set_title("the ohms route: every packet, one mark per hole, the tail windows shaded")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

On the left, the shorted leads settling through the hold, and after the hold the leads moving onto the motor, which takes the reading off the axis. On the right, twelve stretches of forty seconds, one per hole, each starting where the bar was moved and settling into its tail window. A packet off the axis at a mark is the meter reading an open lead while the clamp was loose.

### stall/laps.csv and one stall sweep

Every lap of the stall route, then the first sweep, hole 0 at the lowest level, with the current channel and the two terminal taps over the baseline and the two directions.

In [ ]:
lapdf = pd.read_csv(STALL / "laps.csv")
LEVELS = sorted(float(v) for v in lapdf["volts_set"].unique())
lapdf["trim_mv"] = (lapdf["vsys_read"] - lapdf["volts_set"]) * 1e3                 # how far the trim landed from the level
show(lapdf, n=None)
first = lapdf.iloc[0]
df, meta = tel.load(STALL / first["folder"], BOARD)                                  # checks the sense block against the board entry
LAP_MS = int(str(meta["schedule"][0]).split("@")[1])                                 # the lap's length as the sweep tool ran it
print("segments:", df["seg"].value_counts().sort_index().to_dict(), "| schedule:", meta["schedule"], "| baseline ms:", meta["baseline_ms"],
      "| rest ms:", meta["rest_ms"], "| rows dropped:", meta.get("rows_dropped"))
ts = np.arange(len(df)) / meta["tick_hz"]                                            # time through the sweep; `t` restarts at each segment
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].plot(ts, df["current_raw"], lw=0.5, color=COLOR["stall"]); ax[0].set_ylabel("current_raw (counts)"); ax[0].set_title("the current channel, hole 0")
ax[1].plot(ts, df["vmotor_a"], lw=0.5, color=BLUE, label="tap A"); ax[1].plot(ts, df["vmotor_b"], lw=0.5, color=ORANGE, label="tap B")
ax[1].set_ylabel("terminal taps (counts)"); ax[1].set_title("the taps swap with the direction"); ax[1].legend()
for a in ax:
    a.set_xlabel("time in the sweep (s)"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
missing = {L["folder"]: int(tel.gaps(tel.load(STALL / L["folder"])[0]).sum()) for _, L in lapdf.iterrows()}
print(f"{len(lapdf)} laps at {LEVELS} V, trim within {lapdf['trim_mv'].abs().max():.0f} mV of the level; "
      f"missing rows: {sum(missing.values())} over all laps" + ("" if not any(missing.values()) else f", by lap {missing}"))

Half a second at the zero, then 300 ms each way. The current channel reads the same current in both directions, since the shunt sits in the bridge's common return, and the taps swap. The current climbs through its first milliseconds as the winding's inductance lets it, which the settled mean leaves out.

## Method

### The ohms route

The lead zero is the median of the meter's packets from ten to twenty seconds after the `leads shorted` mark, the last ten seconds of the hold, where the meter has settled on the shorted leads, with the quartiles of the same window as its spread. The window ends with the hold and not at the `hole 0` mark, since the leads move onto the motor between the two and the packets near that mark may read the motor. The lead zero is subtracted from every reading below, so the readings carry the two short leads soldered to the tabs and not the probe leads.

Each hole gives one reading, the median of the meter's packets in the last twenty seconds before the next mark less the lead zero, with the quartiles of the same window as its spread. The first five seconds after the mark are kept apart as the head, and the head less the tail is the settle's fall, how far the reading moved while the contact relaxed after the rotor was turned.

Twelve readings over the angle give a median, a tenth percentile (p10) and a maximum, and the spread between them is what the brushes add on top of the lowest contact. A rule for its shape was pre-registered in earlier work and is applied here as written. The spread is one-sided, a contact that only adds, if the median sits within 1.5 % of p10 and at least 15 % of the holes read more than 3 % over p10. It is two-sided if the median sits more than 1.5 % over p10 and the upper half, p90 less the median, is within a factor of two of the lower half, the median less p10. Anything else is unclassified. The median's 95 % band comes from a bootstrap of 4000 resamples of the twelve readings. With three slots and two brushes the commutator's geometry under the brushes repeats every 60 degrees, so holes two apart see the same geometry, and the readings grouped by the hole's parity say whether the geometry or the seat sets the spread.

### The stall route

Each lap gives two points, one per direction. The chip's reading is the mean of each raw column over the settled part of the direction, which leaves out the first 2 ms after the drive opens, and the current channel's rest zero is the median of `current_raw` over the second half of the lap's own baseline. The counts over that zero go through the current-scale result, counts $= b + G \, I$, so the current is $I = (n - b) / G$ with $G$ the counts per amp and $b$ the in-drive offset. The tap difference, $|tap_A - tap_B|$, goes through the terminal-chain result the same way, $V = (n_{AB} - c) / k$ with $k$ the counts per volt and $c$ its offset, which gives the board's own terminal volts at the connector.

The model is one line through every stall point,

$$V = V_0 + R_s \, I$$

where $V$ is the board's terminal volts at the connector, $I$ the current, $R_s$ the stall resistance and $V_0$ the brush drop, the part of the terminal volts that does not scale with the current. The line is fitted by least squares with the standard errors of both coefficients, over every point, over the graded levels alone, over each level alone and over each direction alone. The lines over each level alone say whether the brush drop holds over the three levels or bends between them. The current-scale and terminal-chain results hold from 3.7 V up and report the 3.3 V level apart, since the board's reference is in dropout there, so the 3.3 V points are drawn open. The pre-registered choice is the line over every point when its slope agrees with the graded line's within their combined standard errors, and the graded line otherwise, with the 3.3 V level reported apart as stage 1 does.

Per hole, the resistance by the stall route is the median over the hole's points of $(V - V_0) / I$ with the chosen line's $V_0$, so one brush drop is taken out of every point and what is left per hole is compared with the ohms route at the same hole. No meter reads this route, so the volts rest on the board's taps as the stage 1 terminal-chain result graded them. The taps sit at the connector, so the stall volts read the motor's two leads as the ohms route does, and the connector's contact on top.

### The ohms route per hole

Each hole's tail median and quartiles less the lead zero, its head, the fall from head to tail, then the twelve readings' spread, the rule and the bootstrap band.

In [ ]:
rows = []
for k, h in holes.iterrows():
    tail = window(dmm_ohms, h["t_next"] - TAIL_S, h["t_next"])["value"].dropna()          # the settled tail, the reading
    head = window(dmm_ohms, h["t_host"], h["t_host"] + HEAD_S)["value"].dropna()           # just after the move
    rows.append({"hole": k, "angle_deg": 30 * k, "n_tail": len(tail), "r_ohm": tail.median() - LEAD_ZERO,
                 "p25": tail.quantile(0.25) - LEAD_ZERO, "p75": tail.quantile(0.75) - LEAD_ZERO, "sd": tail.std(),
                 "head_ohm": head.median() - LEAD_ZERO, "fall_ohm": head.median() - tail.median()})
R = pd.DataFrame(rows)
show(R, n=None)


def dist(x):
    x = np.asarray(x, float)
    return {"median": float(np.median(x)), "p10": float(np.quantile(x, 0.1)), "p90": float(np.quantile(x, 0.9)),
            "min": float(x.min()), "max": float(x.max()), "n": len(x)}


def classify(x):
    # the pre-registered rule, applied as written
    d = dist(x)
    lo = (d["median"] - d["p10"]) / d["p10"]
    share = float(np.mean(np.asarray(x, float) > 1.03 * d["p10"]))
    ratio = (d["p90"] - d["median"]) / (d["median"] - d["p10"]) if d["median"] > d["p10"] else np.inf
    one = lo <= 0.015 and share >= 0.15
    two = lo > 0.015 and 0.5 <= ratio <= 2
    return {"lo_pct": 100 * lo, "share_gt3pct": share, "upper_over_lower": ratio,
            "verdict": "one-sided" if one else "two-sided" if two else "unclassified"}


def boot_median(x, n=4000, seed=1):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float)
    m = [np.median(rng.choice(x, len(x))) for _ in range(n)]
    return [float(v) for v in np.quantile(m, [0.025, 0.975])]


r_holes = R["r_ohm"].to_numpy()
OHMS_D, OHMS_RULE, OHMS_BAND = dist(r_holes), classify(r_holes), boot_median(r_holes)
print(f"ohms route: median {OHMS_D['median']:.3f} ohm (95 % band {OHMS_BAND[0]:.3f} to {OHMS_BAND[1]:.3f}), p10 {OHMS_D['p10']:.3f}, "
      f"max {OHMS_D['max']:.3f}, min {OHMS_D['min']:.3f}; the worst hole {(OHMS_D['max'] / OHMS_D['p10'] - 1) * 100:.1f} % over p10")
print(f"rule: median over p10 {OHMS_RULE['lo_pct']:.2f} %, share of holes over 1.03 x p10 {OHMS_RULE['share_gt3pct']:.2f}, "
      f"upper over lower {OHMS_RULE['upper_over_lower']:.2f} -> {OHMS_RULE['verdict']}")
print(f"inside a tail: sd {R['sd'].min():.3f} to {R['sd'].max():.3f} ohm against the meter's {OHM_CLASS['step']['value']} ohm step; "
      f"fall from head to tail {R['fall_ohm'].min():+.3f} to {R['fall_ohm'].max():+.3f} ohm, {int((R['fall_ohm'] > 0).sum())} of {len(R)} holes fell")
parity = R.groupby(R["hole"] % 2)["r_ohm"].agg(["median", "min", "max"])
print("by the hole's parity, the same commutator geometry every 60 degrees:\n" + parity.round(3).to_string())

One row per hole, the reading less the lead zero with its quartiles, the head and the fall. Under it the five numbers of the twelve, the rule's three inputs and its verdict, and the readings grouped by parity.

The reading against the hole, and every hole's settle from its mark.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={"width_ratios": [1.2, 1]})
ax[0].errorbar(R["hole"], R["r_ohm"], yerr=[R["r_ohm"] - R["p25"], R["p75"] - R["r_ohm"]], fmt="o", color=COLOR["ohms"], capsize=3, label="tail median, quartiles")
ax[0].plot(R["hole"], R["head_ohm"], "x", color=MUTED, label=f"head, first {HEAD_S} s after the move")
ax[0].axhline(OHMS_D["median"], color=INK, lw=0.8, label=f"median {OHMS_D['median']:.3f} ohm")
ax[0].axhline(OHMS_D["p10"], color=INK, lw=0.8, ls=":", label=f"p10 {OHMS_D['p10']:.3f} ohm")
ax[0].axhline(OHMS_D["max"], color=MUTED, lw=0.8, ls="dashed", label=f"max {OHMS_D['max']:.3f} ohm")
ax[0].set_xticks(R["hole"]); ax[0].set_xlabel("hole (30 degrees each)"); ax[0].set_ylabel("meter less the lead zero (ohm)")
ax[0].set_title("the ohms route over the rotor's angle"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
shades = plt.cm.Blues(np.linspace(0.35, 0.95, len(holes)))                           # light to dark with the hole
for shade, (k, h) in zip(shades, holes.iterrows()):
    w = window(dmm_ohms, h["t_host"], h["t_next"])
    ax[1].plot(w["t_host"] - h["t_host"], w["value"] - LEAD_ZERO, lw=0.8, color=shade, label=f"hole {k}")
ax[1].axvspan(holes["t_next"].sub(holes["t_host"]).min() - TAIL_S, holes["t_next"].sub(holes["t_host"]).min(), color=MUTED, alpha=0.15, label="the tail window")
ax[1].set_ylim(OHMS_D["min"] * 0.97, max(OHMS_D["max"], R["head_ohm"].max()) * 1.03)
ax[1].set_xlabel("seconds after the mark"); ax[1].set_ylabel("meter less the lead zero (ohm)"); ax[1].set_title("every settle, one line per hole"); ax[1].legend(fontsize=7, ncol=2); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

Left, one dot per hole with the tail's quartiles as whiskers and the head as a cross, the median, p10 and max drawn across. Right, the meter through each hole's forty seconds from its mark, the tail window shaded, so the settle's shape and how much of the head's excess holds can be seen.

### The stall table

One row per lap and direction, the current and the board's terminal volts through the stage 1 results.

In [ ]:
rows = []
for _, L in lapdf.iterrows():
    df, meta = tel.load(STALL / L["folder"], BOARD)
    zero = float(tel.baseline(df)["current_raw"].median())                             # the lap's own torque-off zero
    for _, g in tel.rungs(df).sort_values("seg").iterrows():                           # the settled means per direction
        rows.append({"hole": int(L["hole"]), "volts_set": L["volts_set"], "vsys_read": L["vsys_read"], "dir": int(g["dir"]), "seg": int(g["seg"]),
                     "zero": zero, "current_raw": g["current_raw"], "current_sd": g["current_raw_sd"], "n": g["n"],
                     "tap_counts": abs(g["vmotor_a"] - g["vmotor_b"]), "vbus_raw": g["vbus_raw"]})
S = pd.DataFrame(rows)
S["amps"] = (S["current_raw"] - S["zero"] - B_OFF) / G_CPA                              # the current-scale line inverted
S["v_board"] = (S["tap_counts"] - C_OFF) / K_CPV                                         # the terminal-chain line inverted, bias free
S["graded"] = S["volts_set"] >= RAIL_GRADED_V
S["joules"] = S["v_board"] * S["amps"] * LAP_MS / 1e3                                    # what one direction puts into the motor
show(S[["hole", "volts_set", "dir", "zero", "current_raw", "current_sd", "amps", "tap_counts", "v_board", "joules"]], n=None)
print(f"{len(S)} points, {S['amps'].min():.3f} to {S['amps'].max():.3f} A, {S['v_board'].min():.3f} to {S['v_board'].max():.3f} V at the connector, "
      f"under {S['joules'].max():.2f} J per direction; current sd inside a direction {S['current_sd'].min():.1f} to {S['current_sd'].max():.1f} counts")

Seventy-two points, three levels and two directions at each of twelve holes.

## Results

### The line through the stall points

The board's terminal volts against the current, every point, with the line over every point and the line over the graded levels, and the residuals under it. The thing to see is whether the 3.3 V points sit on the line the 3.7 V and 4.2 V points make.

In [ ]:
NAN_FIT = {"slope": np.nan, "intercept": np.nan, "slope_se": np.nan, "intercept_se": np.nan, "rms": np.nan, "n": 0}


def fit(g, y="v_board", x="amps"):
    # laps.line over the points that have both, or NaNs when fewer than three
    g = g.dropna(subset=[x, y])
    return laps.line(g[x], g[y]) if len(g) >= 3 else NAN_FIT | {"n": len(g)}


def report(name, g, y="v_board"):
    f = fit(g, y)
    print(f"{name:24s} n={f['n']:2d}  R_s {f['slope']:6.3f} +/- {f['slope_se']:.3f} ohm   V0 {f['intercept']:+6.3f} +/- {f['intercept_se']:.3f} V   rms {f['rms'] * 1e3:5.1f} mV")
    return f

fit_all = report("every point", S)
fit_graded = report(f"{RAIL_GRADED_V:g} V up", S[S["graded"]])
fit_level = {v: report(f"{v:g} V alone", S[S["volts_set"] == v]) for v in LEVELS}
fit_dir = {d: report(f"direction {d:+d}", S[S["dir"] == d]) for d in (1, -1)}
agree = abs(fit_all["slope"] - fit_graded["slope"]) <= np.hypot(fit_all["slope_se"], fit_graded["slope_se"])
fit_stall = fit_all if agree else fit_graded                                           # the pre-registered choice
print(f"\nthe two slopes {'agree' if agree else 'disagree'} within their errors, so the result is the line over "
      f"{'every point' if agree else f'{RAIL_GRADED_V:g} V up'}: R_s {fit_stall['slope']:.3f} ohm, V0 {fit_stall['intercept']:.3f} V")
S["resid_v"] = S["v_board"] - (fit_stall["slope"] * S["amps"] + fit_stall["intercept"])

fig, ax = plt.subplots(2, 1, figsize=(10, 7.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
x = np.linspace(0, S["amps"].max() * 1.05, 10)
for v in LEVELS:
    g = S[S["volts_set"] == v]
    kw = dict(ls="none", marker="o", ms=5, color=COLOR["stall"], mfc=COLOR["stall"] if v >= RAIL_GRADED_V else "none")
    ax[0].plot(g["amps"], g["v_board"], label=f"{v:g} V rail" + ("" if v >= RAIL_GRADED_V else " (not graded in stage 1)"), **kw)
    ax[1].plot(g["amps"], g["resid_v"] * 1e3, **kw)
ax[0].plot(x, fit_all["slope"] * x + fit_all["intercept"], "-", color=INK, lw=1, label=f"every point: R_s {fit_all['slope']:.3f} ohm, V0 {fit_all['intercept']:.3f} V")
ax[0].plot(x, fit_graded["slope"] * x + fit_graded["intercept"], ":", color=INK, lw=1.5, label=f"{RAIL_GRADED_V:g} V up: R_s {fit_graded['slope']:.3f} ohm, V0 {fit_graded['intercept']:.3f} V")
ax[0].set_ylabel("the board's terminal volts at the connector (V)"); ax[0].set_title("the stall line: volts against amps into the locked rotor"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].axhline(0, color=INK, lw=0.5); ax[1].set_ylabel("residual from the chosen line (mV)"); ax[1].set_xlabel("current (A)"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

The points, filled at the graded levels and open at 3.3 V, the line over every point and the line over the graded levels, and under them the residuals from the chosen line. The intercept is the brush drop, the slope the resistance, and the residual panel shows whether the 3.3 V level, the direction or the hole pulls the points off one line. The per-level lines printed above say whether the brush drop holds over the three levels or bends between them.

### Resistance per hole by both routes

Each hole's resistance by the stall route, the brush drop taken out, beside the ohms route at the same hole.

In [ ]:
V0 = fit_stall["intercept"]
S["r_point"] = (S["v_board"] - V0) / S["amps"]                                          # each point with the common brush drop taken out
H = S.groupby("hole").agg(r_stall=("r_point", "median"), r_stall_lo=("r_point", "min"), r_stall_hi=("r_point", "max"), n_points=("r_point", "count"),
                          amps=("amps", "mean")).reset_index()
H = H.merge(R[["hole", "r_ohm", "p25", "p75"]], on="hole")
H["stall_less_ohms"] = H["r_stall"] - H["r_ohm"]
show(H, n=None)
fig, ax = plt.subplots(figsize=(11, 4.4))
ax.errorbar(H["hole"] - 0.1, H["r_ohm"], yerr=[H["r_ohm"] - H["p25"], H["p75"] - H["r_ohm"]], fmt="o", color=COLOR["ohms"], capsize=3, label="ohms route, the meter at a milliamp")
ax.errorbar(H["hole"] + 0.1, H["r_stall"], yerr=[H["r_stall"] - H["r_stall_lo"], H["r_stall_hi"] - H["r_stall"]], fmt="o", color=COLOR["stall"], capsize=3,
            label=f"stall route, (V - V0) / I at under an amp, V0 {V0:.3f} V")
ax.axhline(OHMS_D["median"], color=COLOR["ohms"], lw=0.8, ls=":"); ax.axhline(fit_stall["slope"], color=COLOR["stall"], lw=0.8, ls=":")
ax.set_xticks(H["hole"]); ax.set_xlabel("hole (30 degrees each)"); ax.set_ylabel("resistance (ohm)"); ax.set_title("the two routes over the rotor's angle, the dotted lines their medians")
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
ok = H.dropna(subset=["r_stall"])
corr = float(np.corrcoef(ok["r_ohm"], ok["r_stall"])[0, 1]) if len(ok) >= 3 else np.nan
print(f"stall less ohms per hole: mean {H['stall_less_ohms'].mean():+.3f} ohm, sd {H['stall_less_ohms'].std():.3f}, correlation over the holes {corr:.2f} (n {len(ok)}); "
      f"spread over the holes: ohms route {(OHMS_D['max'] - OHMS_D['min']):.3f} ohm, stall route {(H['r_stall'].max() - H['r_stall'].min()):.3f} ohm")

One pair per hole, the ohms route in blue with the tail's quartiles and the stall route in orange with the hole's six points, three levels and two directions, as the whiskers. If the two routes rise and fall together over the holes the spread is the contact's and both routes read it. If the stall route is flat where the ohms route is not, the contact's excess at a milliamp does not survive at an amp.

### The results file

The ohms route's median with its bootstrap band, p10, max, the per-hole readings and the rule's verdict. The lead zero with its quartiles. The stall line's slope and intercept with their standard errors as the bounds, the line over every point and the graded line beside it, and the per-hole stall resistance.

In [ ]:
q = results.quantity


def se_q(f, key, unit):
    return q(f[key], f[key] - f[key + "_se"], f[key] + f[key + "_se"], unit)


results.write("2-motor", "rest-resistance", CAPTURE, pair=MOTOR.key,
              notes=f"ohms route: the meter on ohms across the tabs, motor off the board, the median of the last {TAIL_S} s at each of twelve holes 30 degrees "
                    f"apart, the lead zero with the leads shorted subtracted; stall route: 100 % laps of {LAP_MS} ms into the locked rotor at {LEVELS} V rails, both directions, "
                    "the current and the terminals through the stage 1 current-scale and terminal-chain results; "
                    f"volts = v0_brush_v + r_stall_ohm x amps over {'every point' if agree else f'the levels from {RAIL_GRADED_V:g} V up'}; "
                    "r_stall_by_hole is the median of (V - V0) / I per hole; rule_verdict is the pre-registered one-sided / two-sided rule on the ohms readings",
              r_ohms_median_ohm=q(OHMS_D["median"], OHMS_BAND[0], OHMS_BAND[1], "ohm"),
              r_ohms_p10_ohm=q(OHMS_D["p10"], unit="ohm"),
              r_ohms_max_ohm=q(OHMS_D["max"], unit="ohm"),
              r_ohms_min_ohm=q(OHMS_D["min"], unit="ohm"),
              r_ohms_spread_pct=q(OHMS_RULE["lo_pct"], unit="%"),
              r_ohms_by_hole=[round(float(v), 4) for v in R["r_ohm"]],
              r_ohms_settle_fall_ohm=q(R["fall_ohm"].median(), R["fall_ohm"].min(), R["fall_ohm"].max(), "ohm"),
              lead_zero_ohm=q(LEAD_ZERO, LEAD_LO, LEAD_HI, "ohm"),
              r_stall_ohm=se_q(fit_stall, "slope", "ohm"),
              v0_brush_v=se_q(fit_stall, "intercept", "V"),
              r_stall_residual_rms_v=q(fit_stall["rms"], unit="V"),
              r_stall_every_point_ohm=se_q(fit_all, "slope", "ohm"),
              v0_every_point_v=se_q(fit_all, "intercept", "V"),
              r_stall_graded_ohm=se_q(fit_graded, "slope", "ohm"),
              v0_graded_v=se_q(fit_graded, "intercept", "V"),
              r_stall_by_hole=[None if not np.isfinite(v) else round(float(v), 4) for v in H["r_stall"]],
              stall_amps_range=q(S["amps"].max(), S["amps"].min(), S["amps"].max(), "A"),
              lap_joules=q(S["joules"].max(), S["joules"].min(), S["joules"].max(), "J"),
              rule_verdict=OHMS_RULE["verdict"])

## Second motor

A second motor of the same batch, read with the meter on ohms at a few holes before the first motor's ohms route, by hand, thirty seconds after each move. There is no dataset for it, so the readings are typed here.

| hole | angle | meter (ohm) |
|-|-|-|
| 0 | 0 degrees | TBD |
| 3 | 90 degrees | TBD |
| 6 | 180 degrees | TBD |
| 9 | 270 degrees | TBD |

To be written from the readings. The paragraph compares the second motor's readings with the first's median and its p10 to max band from the ohms route. Readings that fall inside that band say the two motors sit together and the spread over the angle is larger than the spread between motors of the batch. Readings that sit outside it in one direction say the batch spreads by more than the angle does, and then one motor's resistance is not the family's, and stage 3 and the tools have to read it per servo.

## Limitations

- The meter bounds the ohms route. Its class on the 220 ohm range is 0.5 % of the reading plus 0.1 ohm, from the stage 0 result, and the fixed part is about 2 % of a reading of a few ohms. It moves every hole together and not the spread between holes, which is read in 0.01 ohm steps.
- The leads stay in both routes. The lead zero is read once, at the start, with the probe leads shorted, so the two short leads soldered to the tabs stay in the ohms reading, some milliohms. The stall route's board taps sit at the connector, so they read the motor leads too.
- The 3.3 V level is not graded in stage 1. The board's reference is in dropout there, the rest zero moves, and the lap rows carry no `vref_raw`, so the current and the terminal volts at that level are converted with constants that hold from 3.7 V up.
- The copper warms inside a lap. Under a joule in a direction is about a degree or two on the winding, which moves the copper's resistance by well under a percent, and the twenty seconds of rest per hole are the bench rule and not a measured recovery.
- Twelve holes 30 degrees apart sample the commutator's 60 degree geometry at two phases only, and the bar's holes are not the stops a free rotor picks when a drive ends. The spread the tools will see at a servo's own stops may differ from the spread over the holes.
- Each hole is read once per route. The repeatability of a reading at the same hole after the bar has come off and gone back on is not in this capture.
- The two routes read the contact at two currents, a milliamp and about an amp, with nothing between. A contact whose drop changes with the current reads differently by the two routes, which is what the line separates, but the shape between the two currents is not measured.
- The ohms route runs before the stall route, so the relaxation of the contact after a drive is not in this data. The settle here follows a move of the rotor by hand, not a drive.

## What earlier work established

Before this stage the same measurement was made on an MG90 motor inside its servo, with the meter across the terminals and the rotor stopped by hand and by the drive. Its data is not reused here, but its conclusions bear on what this notebook looks for.

- The reading belongs to the rotor's stop. The bulk of the stops spread about 2 % from p10 to the median and the worst stop read about 20 % over p10, skewed high, which is the shape a contact that only adds would give. Neither set of stops passed the pre-registered rule, so whether the spread is one-sided or two-sided was left open, and the rule is applied here again on twelve indexed stops rather than free ones.
- The reading relaxes after a drive. Every settle fell over its first minute, by about 0.05 ohm plus 15 % of its first-second excess, and most stops that read high in the first second kept more than half their excess through the minute. A contact film relaxing and a thermoelectric voltage at the brushes were the two candidates, and swapping the meter's leads separates them, since a film is symmetric and a thermoelectric voltage changes sign.
- The readings moved 0.3 ohm between two sessions with the motor's first running, its orientation and the hands on the shaft all changed between them. That shift was never explained, and it is the reason the bar holds the pinion here and nothing presses along the shaft.
- A stall slope between two small duties was dominated by the drive windows' own effects at a few microseconds of on time, so a stall resistance is read here at 100 % duty, DC, where nothing switches.

## Open questions

- Whether the settle's fall after a move is a contact film or a thermoelectric voltage. A repeat of two holes with the meter's leads swapped reads the fall again with the meter's current the other way through the brushes. A film gives the same fall, a thermoelectric voltage gives a fall of the other sign.
- Whether the spread over the holes is the commutator's geometry or the seat the bar gives the rotor. A second pass round the circle at the same holes, the bar lifted off and put back at each, gives a second reading per hole. Readings that repeat per hole are the geometry, and readings that scatter by as much as the spread are the seat.
- The 3.3 V level. A lap chunk that carries `vref_raw` beside the current channel gives VDD under the lap, and the 3.3 V points scaled by it either land on the 3.7 V line or leave a remainder that belongs to the board's amplifier at a 3.1 V supply, which is the current-scale notebook's open question as well.
- The relaxation after a drive on this motor. An ohms reading at one hole taken right after the stall route, the meter logging for a minute, reads the fall the earlier work saw and says how long after a drive a resistance reading can be taken on this motor.
- Whether the board's route holds in situ. The board's terminal chain alone carries the stall volts, so the stage 1 terminal-chain result is the only grading they have. A meter on DC V at the tabs through one longer lap at one hole, with the joules it puts in counted, would grade the board's route in situ against the meter.

## Conclusion

To be written from the capture. The conclusion states the terminal resistance at rest by the meter's route as a median over the rotor's angle with its band, its p10 and its maximum, and the rule's verdict on the shape of the spread. It states the stall resistance and the brush drop from the line and which levels' points the line rests on. The stall line rests on the board's own terminal taps and current channel as stage 1 graded them. It then says under which conditions a terminal resistance can be relied on, at a known rotor stop, a known current and a cold winding, and what the teardown and stage 3 take from it. The teardown reads the winding with no brushes in the path and subtracts that copper from `r_ohms_median_ohm` to get the brushes' share. Stage 3 reads the same motor through the gear train and the servo's own stops and is graded against `r_stall_ohm`, `v0_brush_v` and the spread over the holes.